In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Définir le niveau d'optimisation du transpileur

<details>
<summary><b>Versions des packages</b></summary>

Le code de cette page a été développé avec les dépendances suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Les appareils quantiques réels sont soumis au bruit et aux erreurs de portes, c'est pourquoi optimiser les circuits pour réduire leur profondeur et leur nombre de portes peut améliorer significativement les résultats obtenus lors de l'exécution de ces circuits.
La fonction [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) possède un argument positionnel obligatoire, `optimization_level`, qui contrôle l'effort que le transpileur consacre à l'optimisation des circuits. Cet argument est un entier pouvant prendre les valeurs 0, 1, 2 ou 3.
Plus le niveau d'optimisation est élevé, plus les circuits générés sont optimisés, au prix de temps de compilation plus longs.
Le tableau suivant décrit les optimisations effectuées pour chaque valeur.

<table>
  <thead>
    <tr>
      <th>Niveau d'optimisation</th>
      <th>Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        Aucune optimisation : généralement utilisé pour la caractérisation matérielle
        - Traduction de base
        - Layout/Routage : `TrivialLayout`, qui sélectionne les mêmes numéros de qubits physiques que virtuels et insère des SWAPs pour que cela fonctionne (en utilisant `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        Optimisation légère :
        -   Layout/Routage : le layout est d'abord tenté avec `TrivialLayout`. Si des SWAPs supplémentaires sont nécessaires, un layout avec un nombre minimal de SWAPs est trouvé en utilisant `SabreSwap`, puis `VF2LayoutPostLayout` est utilisé pour tenter de sélectionner les meilleurs qubits du graphe.
        -   `InverseCancellation`
        -   Optimisation des portes à 1 qubit
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        Optimisation moyenne :
          - Layout/Routage : niveau d'optimisation 1 (sans trivial) + heuristique optimisée avec une profondeur de recherche et un nombre d'essais plus importants pour la fonction d'optimisation. Comme `TrivialLayout` n'est pas utilisé, il n'y a pas de tentative d'utiliser les mêmes numéros de qubits physiques et virtuels.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        Optimisation élevée :
        - Niveau d'optimisation 2 + heuristique davantage optimisée sur le layout/routage avec plus d'effort et d'essais
        - Resynthèse des blocs à deux qubits en utilisant la [décomposition KAK de Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - Passes rompant l'unitarité :
          * `OptimizeSwapBeforeMeasure` : déplace les mesures pour éviter les SWAPs
          * `RemoveDiagonalGatesBeforeMeasure` : supprime les portes avant les mesures qui n'auraient pas d'effet sur celles-ci
      </td>
    </tr>
  </tbody>
</table>

## Le niveau d'optimisation en pratique
Les portes à deux qubits étant généralement la principale source d'erreurs, on peut quantifier approximativement l'« efficacité matérielle » de la transpilation en comptant le nombre de portes à deux qubits dans le circuit résultant.
Ici, nous allons essayer les différents niveaux d'optimisation sur un circuit d'entrée composé d'une unitaire aléatoire suivie d'une porte SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

Nous utiliserons le backend simulé `FakeSherbrooke` dans nos exemples. Commençons par transpiler avec le niveau d'optimisation 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

Le circuit transpilé comporte six portes ECR à deux qubits.

Recommençons avec le niveau d'optimisation 1 :

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

Le circuit transpilé contient toujours six portes ECR, mais le nombre de portes à un qubit a diminué.

Recommençons avec le niveau d'optimisation 2 :

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

On obtient les mêmes résultats qu'avec le niveau d'optimisation 1. À noter qu'augmenter le niveau d'optimisation ne fait pas toujours une différence.

Recommençons, cette fois avec le niveau d'optimisation 3 :

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

Cette fois, il n'y a plus que trois portes ECR. Ce résultat s'explique par le fait qu'au niveau d'optimisation 3, Qiskit tente de resynthétiser les blocs de portes à deux qubits, et toute porte à deux qubits peut être implémentée avec au plus trois portes ECR. On peut obtenir encore moins de portes ECR en fixant `approximation_degree` à une valeur inférieure à 1, ce qui permet au transpileur de faire des approximations pouvant introduire une légère erreur dans la décomposition des portes (voir [Paramètres couramment utilisés pour la transpilation](common-parameters#approximation-degree)) :

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

Ce circuit ne comporte que deux portes ECR, mais il s'agit d'un circuit approximatif. Pour comprendre en quoi son effet diffère du circuit exact, on peut calculer la fidélité entre l'opérateur unitaire implémenté par ce circuit et l'unitaire exact. Avant d'effectuer le calcul, on réduit d'abord le circuit transpilé, qui contient 127 qubits, à un circuit ne contenant que les qubits actifs, soit deux.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>